# Pointwise Picking vs Continuity Ridge Tracking

This notebook compares two PINN-based extraction methods on the same MAT files:
- Pointwise ridge picking (`extract_ridge`)
- Continuity-based multi-ridge tracking (`continuity_multi_ridge_tracker`)

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path('.').resolve()))

from pinn_dispersion_from_mat import (
    load_pinn_results,
    resample_to_uniform,
    compute_spectrum,
    extract_ridge,
    linear_dispersion,
)
from continuity_multi_ridge_tracker import TrackerConfig, run_tracker_from_mat

In [ ]:
# ---------------- USER SETTINGS ----------------
mat_files = sorted(Path('../Result').glob('*.mat'))

k_coupling = 1.0
mx = 1.0
omega_min = 0.01
skip_transient = 0.20
n_harmonic = 4
pts_per_cycle = 12

tracker_cfg = TrackerConfig(
    n_ridges=3,
    omega_min=omega_min,
    skip_transient=skip_transient,
    n_harmonic=n_harmonic,
    pts_per_cycle=pts_per_cycle,
    max_candidates_per_k=6,
    peak_prominence_db=2.0,
    jump_penalty=1.5,
    suppression_half_width_bins=2,
)

print('MAT files:')
for p in mat_files:
    print(' -', p)

In [ ]:
# Run both methods
pointwise_results = []
continuity_results = []

for p in mat_files:
    # Pointwise
    t_raw, x_raw, params = load_pinn_results(str(p))
    X_raw = x_raw.T

    omega_max_lin = linear_dispersion(np.pi, k_coupling, mx)
    t_u, X_u, dt, omega_nyq = resample_to_uniform(
        t_raw, X_raw, omega_max_lin,
        n_harmonic=n_harmonic,
        pts_per_cycle=pts_per_cycle,
    )
    k_pos, omega_pos, spectrum = compute_spectrum(t_u, X_u, skip_transient=skip_transient)
    k_pw, omega_pw = extract_ridge(k_pos, omega_pos, spectrum, omega_min=omega_min)

    label = Path(p).stem
    if 'phi2' in params:
        label += f" (phi2={params['phi2']:.3g})"

    pointwise_results.append((label, k_pw, omega_pw))

    # Continuity
    cont = run_tracker_from_mat(p, k_coupling=k_coupling, mx=mx, cfg=tracker_cfg)
    continuity_results.append((label, cont['k_pos'], cont['ridges_omega']))

print(f'Processed {len(mat_files)} files.')

In [ ]:
# Side-by-side comparison plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)

# Left: pointwise
ax = axes[0]
for label, k_pw, omega_pw in pointwise_results:
    valid = np.isfinite(omega_pw)
    ax.plot(k_pw[valid] / np.pi, omega_pw[valid], lw=2, label=label)
ax.set_title('Pointwise ridge picking')
ax.set_xlabel(r'Wavenumber $k/\pi$')
ax.set_ylabel(r'Frequency $\omega$ (rad/s)')
ax.grid(alpha=0.25)
ax.legend(fontsize=8)

# Right: continuity
ax = axes[1]
for label, k_c, ridges in continuity_results:
    for i, om in enumerate(ridges, start=1):
        valid = np.isfinite(om)
        ax.plot(k_c[valid] / np.pi, om[valid], lw=2, label=f"{label} - R{i}")
ax.set_title('Continuity-based multi-ridge')
ax.set_xlabel(r'Wavenumber $k/\pi$')
ax.grid(alpha=0.25)
ax.legend(fontsize=7, ncol=2)

plt.suptitle('PINN Dispersion: Pointwise vs Continuity', y=1.02)
plt.tight_layout()
plt.show()